In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface, Probe
from scipy.io import loadmat


probe_data = loadmat("/media/ubuntu/sda/duan/rat/probe/chanMapQPX_mice1.mat")
probe_x = probe_data['xcoords']
probe_y = probe_data['ycoords']

probe_position = pd.DataFrame(probe_x)
probe_position[1] = probe_y

probe = Probe()
probe.set_contacts(positions=probe_position, contact_ids=probe_data['chanMap'][:, 0])


probe.set_device_channel_indices(range(128))



In [2]:
recording_raw = se.read_binary('/home/ubuntu/Downloads/paper/20250615_1.group0.bin', sampling_frequency=30000, dtype=np.int16, num_channels=128*7)
recording_list = []

for i in range(7):
    recording_list.append(recording_raw.select_channels([channel + 128*i for channel in range(128)]))


for i in [0, 1, 2, 3]:
    recording_recorded = spre.bandpass_filter(recording_list[i], freq_min=300, freq_max=3000)
    recording_recorded = spre.notch_filter(recording_recorded, freq=50)
    recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

    recording_f = recording_f.set_probegroup(probe)
    recording_preprocessed = recording_f.save(format="binary")

    output_folder = f'/media/ubuntu/sda/duan/rat/sorting_results/day4/probe_{i+1}'
    os.makedirs(output_folder, exist_ok=True)

    sorting_kilosort4 = ss.run_sorter(sorter_name="kilosort4", recording=recording_preprocessed, folder=output_folder + "/kilosort4")
    analyzer_kilosort4 = si.create_sorting_analyzer(sorting=sorting_kilosort4, recording=recording_preprocessed, format='binary_folder', folder=output_folder + '/analyzer_kilosort4_binary')

    extensions_to_compute = [
        "random_spikes",
        "waveforms",
        "noise_levels",
        "templates",
        "spike_amplitudes",
        "unit_locations",
        "spike_locations",
        "correlograms",
        "template_similarity"
    ]

    extension_params = {
        "unit_locations": {"method": "center_of_mass"},
        "spike_locations": {"ms_before": 0.1},
        "correlograms": {"bin_ms": 0.1},
        "template_similarity": {"method": "cosine_similarity"}
    }

    analyzer_kilosort4.compute(extensions_to_compute, extension_params=extension_params)

    qm_params = sqm.get_default_qm_params()
    analyzer_kilosort4.compute("quality_metrics", qm_params)

    import spikeinterface.exporters as sexp
    sexp.export_to_phy(analyzer_kilosort4, output_folder + "/phy_folder_for_kilosort", verbose=True)
    

AttributeError: module 'spikeinterface.extractors' has no attribute 'read_binary'

In [33]:
recording_raw = se.read_binary('/home/ubuntu/Downloads/paper/20250624_1.group0.bin', sampling_frequency=30000, dtype=np.int16, num_channels=128*7)
recording_list = []

for i in range(7):
    recording_list.append(recording_raw.select_channels([channel + 128*i for channel in range(128)]))


for i in range(7):
    recording_recorded = spre.bandpass_filter(recording_list[i], freq_min=300, freq_max=3000)
    recording_recorded = spre.notch_filter(recording_recorded, freq=50)
    recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

    recording_f = recording_f.set_probegroup(probe)
    recording_preprocessed = recording_f.save(format="binary")

    output_folder = f'/media/ubuntu/sda/duan/rat/sorting_results/day13/probe_{i+1}'
    os.makedirs(output_folder, exist_ok=True)

    sorting_kilosort4 = ss.run_sorter(sorter_name="kilosort4", recording=recording_preprocessed, folder=output_folder + "/kilosort4")
    analyzer_kilosort4 = si.create_sorting_analyzer(sorting=sorting_kilosort4, recording=recording_preprocessed, format='binary_folder', folder=output_folder + '/analyzer_kilosort4_binary')

    extensions_to_compute = [
        "random_spikes",
        "waveforms",
        "noise_levels",
        "templates",
        "spike_amplitudes",
        "unit_locations",
        "spike_locations",
        "correlograms",
        "template_similarity"
    ]

    extension_params = {
        "unit_locations": {"method": "center_of_mass"},
        "spike_locations": {"ms_before": 0.1},
        "correlograms": {"bin_ms": 0.1},
        "template_similarity": {"method": "cosine_similarity"}
    }

    analyzer_kilosort4.compute(extensions_to_compute, extension_params=extension_params)

    qm_params = sqm.get_default_qm_params()
    analyzer_kilosort4.compute("quality_metrics", qm_params)

    import spikeinterface.exporters as sexp
    sexp.export_to_phy(analyzer_kilosort4, output_folder + "/phy_folder_for_kilosort", verbose=True)

Use cache_folder=/tmp/spikeinterface_cache/tmpp_2nvqyz/XP8KCJV8
write_binary_recording 
engine=process - n_jobs=1 - samples_per_chunk=30,000 - chunk_memory=7.32 MiB - total_memory=7.32 MiB - chunk_duration=1.00s


estimate_sparsity (no parallelization): 100%|██████████| 915/915 [00:00<00:00, 16204.96it/s]


create_sorting_analyzer: recording does not have scaling to uV, forcing return_in_uV=False


noise_level (no parallelization): 100%|██████████| 20/20 [00:00<00:00, 33.98it/s]
Compute : spike_amplitudes + spike_locations (no parallelization): 100%|██████████| 915/915 [00:15<00:00, 57.88it/s]
extract PCs (no parallelization): 100%|██████████| 915/915 [15:07<00:00,  1.01it/s]


Run:
phy template-gui  /media/ubuntu/sda/duan/rat/sorting_results/day13/probe_1/phy_folder_for_kilosort/params.py
Use cache_folder=/tmp/spikeinterface_cache/tmphz82j554/253CLR1D
write_binary_recording 
engine=process - n_jobs=1 - samples_per_chunk=30,000 - chunk_memory=7.32 MiB - total_memory=7.32 MiB - chunk_duration=1.00s


...

Run:
phy template-gui  /media/ubuntu/sda/duan/rat/sorting_results/day13/probe_6/phy_folder_for_kilosort/params.py
Use cache_folder=/tmp/spikeinterface_cache/tmpx7x1kaeq/DBILC21Q
write_binary_recording 
engine=process - n_jobs=1 - samples_per_chunk=30,000 - chunk_memory=7.32 MiB - total_memory=7.32 MiB - chunk_duration=1.00s


estimate_sparsity (no parallelization): 100%|██████████| 915/915 [00:00<00:00, 14483.31it/s]


create_sorting_analyzer: recording does not have scaling to uV, forcing return_in_uV=False


noise_level (no parallelization): 100%|██████████| 20/20 [00:00<00:00, 35.07it/s]
Compute : spike_amplitudes + spike_locations (no parallelization): 100%|██████████| 915/915 [00:12<00:00, 70.96it/s]
extract PCs (no parallelization): 100%|██████████| 915/915 [15:56<00:00,  1.05s/it]

Run:
phy template-gui  /media/ubuntu/sda/duan/rat/sorting_results/day13/probe_7/phy_folder_for_kilosort/params.py
